In [ ]:
!pip install feedparser

In [ ]:
import os
import sys
import json
import argparse
import logging
import re
import asyncio
import traceback
import csv
import time
import requests
import feedparser
from pathlib import Path
from typing import Optional
from urllib.parse import quote

import anthropic
from anthropic import AsyncAnthropic
import pdfplumber
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

# =============================================================================
# CONFIGURATION - Edit these settings and just run the script!
# =============================================================================

CONFIG = {
    # ---------- Paper Discovery Settings ----------
    # How to find papers. Options: "local", "inspire", "arxiv", "both_apis"
    #   "local"     = Read PDFs from input_folder (original behavior)
    #   "inspire"   = Query INSPIRE HEP API for papers citing CMS Open Data
    #   "arxiv"     = Query arXiv API for papers mentioning "CMS Open Data"
    #   "both_apis" = Query both INSPIRE and arXiv, merge & deduplicate
    "paper_source": "both_apis",

    # Custom search queries (used when paper_source is "inspire", "arxiv", or "both_apis")
    # INSPIRE query: finds papers that cite CMS Open Data DOIs
    "inspire_query": "references.reference.dois:10.7483/OPENDATA.CMS*",
    # arXiv search term: finds papers with this in title/abstract
    "arxiv_search_query": '"CMS Open Data"',

    # Maximum number of papers to fetch from each API (None = all available)
    "max_papers_per_api": None,

    # ---------- File / Folder Paths ----------
    # Path to folder containing your PDF papers (used when paper_source = "local")
    "input_folder": r"C:/Users/ejren/OneDrive/DPOA_papers",

    # Folder where downloaded PDFs will be saved
    "download_folder": r"C:/Users/ejren/OneDrive/DPOA_papers/downloaded",

    # Output CSV file path
    "output_file": "dataset_infoV4.csv",

    # ---------- API & Processing Settings ----------
    # Anthropic API key (or set ANTHROPIC_API_KEY environment variable)
    "api_key": None,

    # Model to use
    "model": "claude-sonnet-4-20250514",

    # Maximum pages to process per PDF (None = all pages)
    "max_pages": None,

    # Maximum tokens per API call chunk
    "max_tokens_per_chunk": 15000,

    # Seconds to wait between papers (increase if hitting rate limits)
    "delay_between_papers": 5,

    # Skip papers that have already been downloaded
    "skip_existing_downloads": True,

    # Logging level: "quiet", "normal", or "verbose"
    "log_level": "normal",
}

# =============================================================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# =============================================================================
# PAPER DISCOVERY - INSPIRE HEP & arXiv APIs
# (Based on cms-dpoa/data-usage: github.com/mattbellis/data-usage)
# =============================================================================

def query_inspire_api(
    query: str = "references.reference.dois:10.7483/OPENDATA.CMS*",
    max_papers: Optional[int] = None,
    page_size: int = 25,
) -> list[dict]:
    """
    Query the INSPIRE HEP API for papers matching the given query.
    Returns a list of paper metadata dicts with keys:
      title, authors, arxiv_id, doi, inspire_id, year, abstract, pdf_url
    
    Based on the query pattern from cms-dpoa/data-usage repo.
    API docs: https://github.com/inspirehep/rest-api-doc
    """
    logger.info(f"Querying INSPIRE HEP API: {query}")
    base_url = "https://inspirehep.net/api/literature"
    papers = []
    page = 1

    while True:
        params = {
            "sort": "mostrecent",
            "size": page_size,
            "page": page,
            "q": query,
        }

        try:
            resp = requests.get(base_url, params=params, timeout=30)
            resp.raise_for_status()
            data = resp.json()
        except requests.RequestException as e:
            logger.error(f"INSPIRE API request failed (page {page}): {e}")
            break

        hits = data.get("hits", {}).get("hits", [])
        if not hits:
            break

        for hit in hits:
            meta = hit.get("metadata", {})
            paper = _parse_inspire_record(meta)
            if paper:
                papers.append(paper)

        total = data.get("hits", {}).get("total", 0)
        logger.info(f"  INSPIRE page {page}: got {len(hits)} hits (total: {total})")

        if max_papers and len(papers) >= max_papers:
            papers = papers[:max_papers]
            break

        if len(papers) >= total:
            break

        page += 1
        time.sleep(1)  # Be nice to the API (rate limit: 15 req per 5s)

    logger.info(f"INSPIRE: found {len(papers)} papers total")
    return papers


def _parse_inspire_record(meta: dict) -> Optional[dict]:
    """Parse an INSPIRE metadata record into a standardized paper dict."""
    # Title
    titles = meta.get("titles", [])
    title = titles[0].get("title", "") if titles else ""

    # Authors
    authors_list = meta.get("authors", [])
    authors = "; ".join(a.get("full_name", "") for a in authors_list[:20])
    if len(authors_list) > 20:
        authors += "; et al."

    # arXiv ID
    arxiv_eprints = meta.get("arxiv_eprints", [])
    arxiv_id = arxiv_eprints[0].get("value", "") if arxiv_eprints else ""

    # DOI
    dois = meta.get("dois", [])
    doi = dois[0].get("value", "") if dois else ""

    # Year
    earliest_date = meta.get("earliest_date", "")
    year = earliest_date[:4] if earliest_date else ""

    # Abstract
    abstracts = meta.get("abstracts", [])
    abstract = abstracts[0].get("value", "") if abstracts else ""

    # INSPIRE record ID
    inspire_id = str(meta.get("control_number", ""))

    # PDF URL (from arXiv if available)
    pdf_url = f"https://arxiv.org/pdf/{arxiv_id}" if arxiv_id else ""

    if not title:
        return None

    return {
        "title": title,
        "authors": authors,
        "arxiv_id": arxiv_id,
        "doi": doi,
        "inspire_id": inspire_id,
        "year": year,
        "abstract": abstract,
        "pdf_url": pdf_url,
        "source": "inspire",
    }


def query_arxiv_api(
    search_query: str = '"CMS Open Data"',
    max_papers: Optional[int] = None,
    batch_size: int = 50,
) -> list[dict]:
    """
    Query the arXiv API for papers matching the search query.
    Returns a list of paper metadata dicts.
    
    Based on the arXiv query pattern from cms-dpoa/data-usage repo.
    Uses feedparser to parse the Atom XML response.
    API docs: https://info.arxiv.org/help/api/basics.html
    """
    logger.info(f"Querying arXiv API: {search_query}")
    base_url = "https://export.arxiv.org/api/query"
    papers = []
    start = 0

    while True:
        params = {
            "search_query": f"all:{search_query}",
            "start": start,
            "max_results": batch_size,
            "sortBy": "submittedDate",
            "sortOrder": "descending",
        }

        try:
            resp = requests.get(base_url, params=params, timeout=30)
            resp.raise_for_status()
            feed = feedparser.parse(resp.text)
        except requests.RequestException as e:
            logger.error(f"arXiv API request failed (start={start}): {e}")
            break

        entries = feed.get("entries", [])
        if not entries:
            break

        for entry in entries:
            paper = _parse_arxiv_entry(entry)
            if paper:
                papers.append(paper)

        total_results = int(feed.feed.get("opensearch_totalresults", 0))
        logger.info(f"  arXiv batch start={start}: got {len(entries)} entries (total: {total_results})")

        if max_papers and len(papers) >= max_papers:
            papers = papers[:max_papers]
            break

        start += batch_size
        if start >= total_results:
            break

        time.sleep(3)  # arXiv asks for 3s delay between requests

    logger.info(f"arXiv: found {len(papers)} papers total")
    return papers


def _parse_arxiv_entry(entry: dict) -> Optional[dict]:
    """Parse an arXiv feed entry into a standardized paper dict."""
    title = entry.get("title", "").replace("\n", " ").strip()
    title = re.sub(r'\s+', ' ', title)

    authors = "; ".join(a.get("name", "") for a in entry.get("authors", []))

    # Extract arXiv ID from the entry ID URL
    entry_id = entry.get("id", "")
    arxiv_id = ""
    if "arxiv.org/abs/" in entry_id:
        arxiv_id = entry_id.split("arxiv.org/abs/")[-1]
        # Remove version suffix for the base ID
        arxiv_id_base = re.sub(r'v\d+$', '', arxiv_id)
    else:
        arxiv_id_base = arxiv_id

    doi = entry.get("arxiv_doi", "")

    published = entry.get("published", "")
    year = published[:4] if published else ""

    abstract = entry.get("summary", "").replace("\n", " ").strip()
    abstract = re.sub(r'\s+', ' ', abstract)

    # PDF link
    pdf_url = ""
    for link in entry.get("links", []):
        if link.get("type") == "application/pdf":
            pdf_url = link.get("href", "")
            break
    if not pdf_url and arxiv_id_base:
        pdf_url = f"https://arxiv.org/pdf/{arxiv_id_base}"

    if not title:
        return None

    return {
        "title": title,
        "authors": authors,
        "arxiv_id": arxiv_id_base or arxiv_id,
        "doi": doi,
        "inspire_id": "",
        "year": year,
        "abstract": abstract,
        "pdf_url": pdf_url,
        "source": "arxiv",
    }


def discover_papers(
    paper_source: str,
    inspire_query: str,
    arxiv_search_query: str,
    max_papers_per_api: Optional[int],
) -> list[dict]:
    """
    Discover papers from the configured source(s).
    Deduplicates by arXiv ID when using both APIs.
    """
    papers = []

    if paper_source in ("inspire", "both_apis"):
        inspire_papers = query_inspire_api(inspire_query, max_papers_per_api)
        papers.extend(inspire_papers)

    if paper_source in ("arxiv", "both_apis"):
        arxiv_papers = query_arxiv_api(arxiv_search_query, max_papers_per_api)
        papers.extend(arxiv_papers)

    # Deduplicate by arXiv ID (prefer INSPIRE records as they have more metadata)
    if paper_source == "both_apis":
        seen_arxiv = {}
        unique_papers = []
        for p in papers:
            aid = p.get("arxiv_id", "")
            if aid:
                if aid not in seen_arxiv:
                    seen_arxiv[aid] = p
                    unique_papers.append(p)
                elif p["source"] == "inspire" and seen_arxiv[aid]["source"] == "arxiv":
                    # Replace arxiv-only record with richer inspire record
                    idx = unique_papers.index(seen_arxiv[aid])
                    unique_papers[idx] = p
                    seen_arxiv[aid] = p
            else:
                unique_papers.append(p)
        papers = unique_papers

    logger.info(f"Total unique papers discovered: {len(papers)}")
    return papers


def download_paper_pdf(
    paper: dict,
    download_folder: Path,
    skip_existing: bool = True,
) -> Optional[Path]:
    """
    Download a paper's PDF from arXiv.
    Returns the local file path, or None if download failed.
    """
    pdf_url = paper.get("pdf_url", "")
    if not pdf_url:
        logger.warning(f"No PDF URL for: {paper.get('title', 'unknown')}")
        return None

    # Create a safe filename from the arXiv ID or title
    arxiv_id = paper.get("arxiv_id", "")
    if arxiv_id:
        safe_name = arxiv_id.replace("/", "_").replace(".", "_")
    else:
        safe_name = re.sub(r'[^\w\s-]', '', paper.get("title", "unknown"))[:80]
        safe_name = re.sub(r'\s+', '_', safe_name)

    filename = f"{safe_name}.pdf"
    filepath = download_folder / filename

    if skip_existing and filepath.exists() and filepath.stat().st_size > 1000:
        logger.info(f"  Already downloaded: {filename}")
        return filepath

    logger.info(f"  Downloading: {pdf_url}")
    try:
        resp = requests.get(pdf_url, timeout=60, stream=True)
        resp.raise_for_status()

        # Check that we actually got a PDF
        content_type = resp.headers.get("content-type", "")
        if "html" in content_type.lower() and "pdf" not in content_type.lower():
            logger.warning(f"  Got HTML instead of PDF for {filename}, skipping")
            return None

        with open(filepath, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)

        size = filepath.stat().st_size
        if size < 1000:
            logger.warning(f"  Downloaded file too small ({size} bytes): {filename}")
            filepath.unlink(missing_ok=True)
            return None

        logger.info(f"  Saved: {filename} ({size:,} bytes)")
        return filepath

    except requests.RequestException as e:
        logger.error(f"  Download failed for {filename}: {e}")
        return None


def download_all_papers(
    papers: list[dict],
    download_folder: Path,
    skip_existing: bool = True,
    delay: float = 1.0,
) -> list[Path]:
    """Download PDFs for all discovered papers. Returns list of local paths."""
    download_folder.mkdir(parents=True, exist_ok=True)
    downloaded = []

    for i, paper in enumerate(papers):
        logger.info(f"Downloading ({i+1}/{len(papers)}): {paper.get('title', '')[:60]}...")
        path = download_paper_pdf(paper, download_folder, skip_existing)
        if path:
            downloaded.append(path)
        if i < len(papers) - 1:
            time.sleep(delay)

    logger.info(f"Downloaded {len(downloaded)}/{len(papers)} papers")
    return downloaded


# =============================================================================
# DATASET EXTRACTION (your original code, unchanged)
# =============================================================================

DATASET_EXTRACTION_TOOL = {
    "name": "extract_datasets",
    "description": "Extract all dataset information found in the physics paper",
    "input_schema": {
        "type": "object",
        "properties": {
            "datasets": {
                "type": "array",
                "description": "List of datasets found in the paper",
                "items": {
                    "type": "object",
                    "properties": {
                        "full_paper_name": {
                            "type": "string",
                            "description": "Full title of the paper including any subtitle"
                        },
                        "shortened_paper_name": {
                            "type": "string",
                            "description": "Short version of the paper title (main title only, no subtitle)"
                        },
                        "year_published": {
                            "type": "string",
                            "description": "Year the paper was published (e.g. 2024)"
                        },
                        "journal": {
                            "type": "string",
                            "description": "Journal or conference (e.g. arXiv, Physical Review Letters, JHEP)"
                        },
                        "authors": {
                            "type": "string",
                            "description": "Authors semicolon-separated (e.g. John Doe; Jane Smith)"
                        },
                        "dataset_name": {
                            "type": "string",
                            "description": "Name or identifier of the dataset"
                        },
                        "dataset_type": {
                            "type": "string",
                            "enum": ["Real Data", "Simulated MC"],
                            "description": "Whether this is real collision data or Monte Carlo simulation"
                        },
                        "official_path": {
                            "type": "string",
                            "description": "Official dataset path e.g. /Jet/Run2010B-Apr21ReReco-v1/AOD"
                        },
                        "events_total": {
                            "type": "string",
                            "description": "Total number of events in the dataset, or null if unknown"
                        },
                        "events_used": {
                            "type": "string",
                            "description": "Number of events used in the analysis, or null if unknown"
                        },
                        "collision_energy_tev": {
                            "type": "string",
                            "description": "Collision energy in TeV (e.g. 13, 7, 8)"
                        },
                        "generator": {
                            "type": "string",
                            "description": "MC generator used (e.g. PYTHIA, MadGraph) or N/A for real data"
                        },
                        "doi": {
                            "type": "string",
                            "description": "DOI identifier for the dataset if available"
                        },
                        "size_bytes": {
                            "type": "string",
                            "description": "Dataset size if mentioned (e.g. 1TB=1e12, 1GB=1e9)"
                        },
                        "luminosity": {
                            "type": "string",
                            "description": "Integrated luminosity (e.g. 5.1 fb-1)"
                        },
                        "notes": {
                            "type": "string",
                            "description": "Any other relevant details about the dataset"
                        }
                    },
                    "required": ["dataset_name", "dataset_type"]
                }
            }
        },
        "required": ["datasets"]
    }
}

EXTRACTION_PROMPT = """You are analyzing a physics paper. Extract ALL datasets mentioned in this paper.

For EVERY dataset (both real collision data AND Monte Carlo simulations), extract:
- full_paper_name: the complete title of the paper
- shortened_paper_name: a shorter version of the title
- year_published: year of publication
- journal: where it was published (arXiv, PRL, etc.)
- authors: all authors semicolon-separated
- dataset_name: name or identifier of the dataset
- dataset_type: "Real Data" or "Simulated MC"
- official_path: CMS/ATLAS dataset path if given
- events_total: total events in dataset
- events_used: events used in the analysis
- collision_energy_tev: collision energy in TeV
- generator: MC generator name or N/A for real data
- doi: dataset DOI if given
- size_bytes: dataset size if mentioned
- luminosity: integrated luminosity if mentioned
- notes: any other useful details

Use "null" for any field not found in the paper. Do not skip datasets.

Paper text:
{paper_text}
"""


# -----------------------------------------------------------------------------
# Text helpers (unchanged from original)
# -----------------------------------------------------------------------------

def preprocess_text(text: str) -> str:
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\t+', ' ', text)
    lines = text.split('\n')
    if len(lines) > 20:
        line_counts = {}
        for line in lines:
            stripped = line.strip()
            if len(stripped) > 10:
                line_counts[stripped] = line_counts.get(stripped, 0) + 1
        repeated = {line for line, count in line_counts.items() if count > 3}
        lines = [line for line in lines if line.strip() not in repeated]
        text = '\n'.join(lines)
    return text.strip()


def chunk_text(text: str, max_tokens: int = 15000) -> list[str]:
    max_chars = max_tokens * 4
    if len(text) <= max_chars:
        return [text]
    chunks = []
    paragraphs = re.split(r'\n\s*\n', text)
    current_chunk = []
    current_chars = 0
    for para in paragraphs:
        para_chars = len(para)
        if current_chars + para_chars > max_chars and current_chunk:
            chunks.append('\n\n'.join(current_chunk))
            current_chunk = [para]
            current_chars = para_chars
        else:
            current_chunk.append(para)
            current_chars += para_chars
    if current_chunk:
        chunks.append('\n\n'.join(current_chunk))
    return chunks


# -----------------------------------------------------------------------------
# PDF extraction (unchanged from original)
# -----------------------------------------------------------------------------

def extract_text_from_pdf(pdf_path: Path, max_pages: Optional[int] = None) -> str:
    logger.info(f"Extracting text from: {pdf_path.name}")
    text_parts = []
    with pdfplumber.open(pdf_path) as pdf:
        pages = pdf.pages[:max_pages] if max_pages else pdf.pages
        for i, page in enumerate(pages):
            page_text = page.extract_text()
            if page_text:
                text_parts.append(f"--- Page {i+1} ---\n{page_text}")
            for j, table in enumerate(page.extract_tables() or []):
                if table:
                    table_text = "\n".join(
                        "\t".join(str(cell) if cell else "" for cell in row)
                        for row in table
                    )
                    text_parts.append(f"--- Table {j+1} on Page {i+1} ---\n{table_text}")
    full_text = "\n\n".join(text_parts)
    logger.info(f"Extracted {len(full_text):,} characters from {len(pages)} pages")
    return full_text


# -----------------------------------------------------------------------------
# API calls (unchanged from original)
# -----------------------------------------------------------------------------

async def call_api(
    client: AsyncAnthropic,
    prompt: str,
    model: str,
    paper_name: str,
    chunk_label: str,
    max_retries: int = 5,
    retry_delay: float = 120.0
) -> list[dict]:
    for attempt in range(max_retries):
        try:
            message = await client.messages.create(
                model=model,
                max_tokens=4096,
                tools=[DATASET_EXTRACTION_TOOL],
                tool_choice={"type": "tool", "name": "extract_datasets"},
                messages=[{"role": "user", "content": prompt}]
            )
            datasets = []
            for block in message.content:
                if block.type == "tool_use" and block.name == "extract_datasets":
                    extracted = block.input.get("datasets", [])
                    for ds in extracted:
                        ds["paper"] = paper_name
                    datasets.extend(extracted)
            logger.info(f"  {chunk_label}: extracted {len(datasets)} datasets")
            return datasets

        except anthropic.RateLimitError:
            if attempt < max_retries - 1:
                wait = retry_delay * (attempt + 1)
                logger.warning(f"  Rate limit on {chunk_label}, waiting {wait:.0f}s (attempt {attempt+1}/{max_retries})")
                await asyncio.sleep(wait)
            else:
                logger.error(f"  Rate limit exceeded after {max_retries} attempts for {chunk_label}")
                return []
        except anthropic.BadRequestError as e:
            logger.error(f"  Bad request for {chunk_label}: {e}")
            return []
        except Exception as e:
            logger.error(f"  Unexpected error for {chunk_label}: {type(e).__name__}: {e}")
            return []
    return []


async def process_paper(
    pdf_path: Path,
    client: AsyncAnthropic,
    model: str,
    max_pages: Optional[int],
    max_tokens_per_chunk: int,
) -> list[dict]:
    try:
        raw_text = extract_text_from_pdf(pdf_path, max_pages)
        paper_name = pdf_path.stem
        clean_text = preprocess_text(raw_text)
        chunks = chunk_text(clean_text, max_tokens=max_tokens_per_chunk)

        if len(chunks) > 1:
            logger.info(f"  Split into {len(chunks)} chunks")

        all_datasets = []
        for i, chunk in enumerate(chunks, 1):
            label = f"{paper_name} chunk {i}/{len(chunks)}"
            prompt = EXTRACTION_PROMPT.format(paper_text=chunk)
            datasets = await call_api(client, prompt, model, paper_name, label)
            all_datasets.extend(datasets)
            if i < len(chunks):
                await asyncio.sleep(3)

        seen = set()
        unique = []
        for ds in all_datasets:
            if not isinstance(ds, dict):
                continue
            key = (
                ds.get("dataset_name", "") or "",
                ds.get("official_path", "") or "",
                ds.get("doi", "") or ""
            )
            if key not in seen:
                seen.add(key)
                unique.append(ds)

        logger.info(f"Finished {pdf_path.name}: {len(unique)} unique datasets found")
        return unique

    except Exception as e:
        logger.error(f"Failed to process {pdf_path.name}: {e}")
        logger.debug(traceback.format_exc())
        return []


# -----------------------------------------------------------------------------
# Main orchestration (updated to support both local and API-discovered papers)
# -----------------------------------------------------------------------------

def find_pdfs(folder: Path) -> list[Path]:
    all_pdfs = list(folder.glob("*.pdf")) + list(folder.glob("*.PDF"))
    seen_stems = {}
    for f in all_pdfs:
        key = f.stem.lower()
        if key not in seen_stems:
            seen_stems[key] = f
    return list(seen_stems.values())


async def run_extraction(
    pdf_files: list[Path],
    api_key: Optional[str],
    model: str,
    max_pages: Optional[int],
    max_tokens_per_chunk: int,
    delay_between_papers: int,
) -> list[dict]:
    client = AsyncAnthropic(api_key=api_key) if api_key else AsyncAnthropic()
    all_datasets = []

    for i, pdf_path in enumerate(pdf_files):
        logger.info(f"\n{'='*60}")
        logger.info(f"Processing ({i+1}/{len(pdf_files)}): {pdf_path.name}")
        logger.info(f"{'='*60}")

        datasets = await process_paper(
            pdf_path=pdf_path,
            client=client,
            model=model,
            max_pages=max_pages,
            max_tokens_per_chunk=max_tokens_per_chunk,
        )
        all_datasets.extend(datasets)

        if i < len(pdf_files) - 1:
            logger.info(f"Waiting {delay_between_papers}s before next paper...")
            await asyncio.sleep(delay_between_papers)

    return all_datasets


def save_results(datasets: list[dict], output_file: str) -> pd.DataFrame:
    columns = [
        "paper",
        "full_paper_name",
        "shortened_paper_name",
        "year_published",
        "journal",
        "authors",
        "dataset_name",
        "dataset_type",
        "official_path",
        "events_total",
        "events_used",
        "collision_energy_tev",
        "generator",
        "doi",
        "size_bytes",
        "luminosity",
        "notes",
    ]
    df = pd.DataFrame(datasets)
    for col in columns:
        if col not in df.columns:
            df[col] = "N/A"
    df = df[columns]
    df.to_csv(output_file, index=False, quoting=csv.QUOTE_ALL)
    logger.info(f"Saved {len(df)} rows to {output_file}")
    return df


def save_paper_index(papers: list[dict], output_path: str = "discovered_papers.json"):
    """Save the list of discovered papers as a JSON index for reference."""
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(papers, f, indent=2, ensure_ascii=False)
    logger.info(f"Saved paper index ({len(papers)} papers) to {output_path}")


def process_papers(
    paper_source: str = "local",
    input_folder: str = "",
    download_folder: str = "",
    output_file: str = "dataset_info.csv",
    inspire_query: str = "references.reference.dois:10.7483/OPENDATA.CMS*",
    arxiv_search_query: str = '"CMS Open Data"',
    max_papers_per_api: Optional[int] = None,
    skip_existing_downloads: bool = True,
    api_key: Optional[str] = None,
    model: str = "claude-sonnet-4-20250514",
    max_pages: Optional[int] = None,
    max_tokens_per_chunk: int = 15000,
    delay_between_papers: int = 5,
) -> pd.DataFrame:
    """
    Main entry point. Discovers papers, downloads PDFs, and extracts datasets.
    
    paper_source options:
      "local"     - Use PDFs already in input_folder
      "inspire"   - Query INSPIRE HEP API, download PDFs, then extract
      "arxiv"     - Query arXiv API, download PDFs, then extract
      "both_apis" - Query both APIs, deduplicate, download, then extract
    """

    if paper_source == "local":
        # Original behavior: read from local folder
        input_path = Path(input_folder)
        if not input_path.exists():
            raise FileNotFoundError(f"Input folder not found: {input_folder}")
        pdf_files = find_pdfs(input_path)
        if not pdf_files:
            logger.warning(f"No PDF files found in {input_folder}")
            return pd.DataFrame()
        logger.info(f"Found {len(pdf_files)} local PDF files to process")

    else:
        # NEW: Discover papers via APIs, download PDFs
        papers = discover_papers(
            paper_source=paper_source,
            inspire_query=inspire_query,
            arxiv_search_query=arxiv_search_query,
            max_papers_per_api=max_papers_per_api,
        )

        if not papers:
            logger.warning("No papers discovered from APIs")
            return pd.DataFrame()

        # Save paper index for reference
        save_paper_index(papers)

        # Filter to papers that have a PDF URL
        papers_with_pdf = [p for p in papers if p.get("pdf_url")]
        logger.info(f"{len(papers_with_pdf)}/{len(papers)} papers have PDF URLs")

        if not papers_with_pdf:
            logger.warning("No papers with downloadable PDFs found")
            return pd.DataFrame()

        # Download PDFs
        dl_folder = Path(download_folder)
        pdf_files = download_all_papers(
            papers_with_pdf, dl_folder, skip_existing_downloads
        )

        if not pdf_files:
            logger.warning("No PDFs were successfully downloaded")
            return pd.DataFrame()

    # Run extraction pipeline
    all_datasets = asyncio.run(run_extraction(
        pdf_files=pdf_files,
        api_key=api_key,
        model=model,
        max_pages=max_pages,
        max_tokens_per_chunk=max_tokens_per_chunk,
        delay_between_papers=delay_between_papers,
    ))

    if not all_datasets:
        logger.warning("No datasets extracted from any papers")
        return pd.DataFrame()

    return save_results(all_datasets, output_file)


# -----------------------------------------------------------------------------
# Entry points
# -----------------------------------------------------------------------------

def run_with_config():
    """Run using CONFIG dict at top of file."""
    if CONFIG["log_level"] == "quiet":
        logging.getLogger().setLevel(logging.WARNING)
    elif CONFIG["log_level"] == "verbose":
        logging.getLogger().setLevel(logging.DEBUG)

    api_key = CONFIG["api_key"] or os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        print("ERROR: No API key. Set ANTHROPIC_API_KEY or add it to CONFIG.")
        return None

    source = CONFIG["paper_source"]

    print(f"{'='*60}")
    print("DATASET EXTRACTION FROM PHYSICS PAPERS")
    print(f"{'='*60}")
    print(f"Paper source : {source}")
    if source == "local":
        print(f"Input folder : {CONFIG['input_folder']}")
    else:
        print(f"Download to  : {CONFIG['download_folder']}")
        if source in ("inspire", "both_apis"):
            print(f"INSPIRE query: {CONFIG['inspire_query']}")
        if source in ("arxiv", "both_apis"):
            print(f"arXiv query  : {CONFIG['arxiv_search_query']}")
        if CONFIG["max_papers_per_api"]:
            print(f"Max papers   : {CONFIG['max_papers_per_api']} per API")
    print(f"Output file  : {CONFIG['output_file']}")
    print(f"Chunk size   : {CONFIG['max_tokens_per_chunk']} tokens")
    print(f"Paper delay  : {CONFIG['delay_between_papers']}s")
    print(f"{'='*60}\n")

    try:
        df = process_papers(
            paper_source=CONFIG["paper_source"],
            input_folder=CONFIG["input_folder"],
            download_folder=CONFIG["download_folder"],
            output_file=CONFIG["output_file"],
            inspire_query=CONFIG["inspire_query"],
            arxiv_search_query=CONFIG["arxiv_search_query"],
            max_papers_per_api=CONFIG["max_papers_per_api"],
            skip_existing_downloads=CONFIG["skip_existing_downloads"],
            api_key=api_key,
            model=CONFIG["model"],
            max_pages=CONFIG["max_pages"],
            max_tokens_per_chunk=CONFIG["max_tokens_per_chunk"],
            delay_between_papers=CONFIG["delay_between_papers"],
        )

        if not df.empty:
            print(f"\n{'='*60}")
            print("EXTRACTION COMPLETE")
            print(f"{'='*60}")
            print(f"Total datasets extracted : {len(df)}")
            print(f"Unique papers processed  : {df['paper'].nunique()}")
            print(f"Output saved to          : {CONFIG['output_file']}")
            print(f"\nDataset types:")
            print(df['dataset_type'].value_counts().to_string())
            print(f"\nFirst few rows:")
            print(df[["paper", "full_paper_name", "dataset_name", "dataset_type"]].head(10).to_string())
        else:
            print("No datasets were extracted.")

        return df

    except Exception as e:
        print(f"ERROR: {e}")
        traceback.print_exc()
        return None


def main():
    parser = argparse.ArgumentParser(
        description="Discover and extract dataset info from physics papers"
    )
    parser.add_argument("--source", "-s", default="local",
                        choices=["local", "inspire", "arxiv", "both_apis"],
                        help="Where to find papers (default: local)")
    parser.add_argument("--input_folder", "-i", default=None,
                        help="Folder with local PDFs (for --source local)")
    parser.add_argument("--download_folder", "-d", default="./downloaded_papers",
                        help="Folder to save downloaded PDFs")
    parser.add_argument("--output", "-o", default="dataset_info.csv")
    parser.add_argument("--inspire_query", default="references.reference.dois:10.7483/OPENDATA.CMS*",
                        help="INSPIRE HEP search query")
    parser.add_argument("--arxiv_query", default='"CMS Open Data"',
                        help="arXiv search query")
    parser.add_argument("--max_papers", type=int, default=None,
                        help="Max papers to fetch per API")
    parser.add_argument("--api_key", "-k", default=None)
    parser.add_argument("--model", "-m", default="claude-sonnet-4-20250514")
    parser.add_argument("--max_pages", type=int, default=None)
    parser.add_argument("--max_tokens", type=int, default=15000)
    parser.add_argument("--delay", type=int, default=5)
    parser.add_argument("--quiet", "-q", action="store_true")
    parser.add_argument("--verbose", "-v", action="store_true")
    parser.add_argument("--discover_only", action="store_true",
                        help="Only discover and download papers, skip extraction")
    args = parser.parse_args()

    if args.quiet:
        logging.getLogger().setLevel(logging.WARNING)
    elif args.verbose:
        logging.getLogger().setLevel(logging.DEBUG)

    if args.source == "local" and not args.input_folder:
        logger.error("--input_folder is required when --source is 'local'")
        sys.exit(1)

    # For discover-only mode: just find and download papers
    if args.discover_only:
        papers = discover_papers(
            paper_source=args.source,
            inspire_query=args.inspire_query,
            arxiv_search_query=args.arxiv_query,
            max_papers_per_api=args.max_papers,
        )
        save_paper_index(papers)
        if papers:
            dl_folder = Path(args.download_folder)
            downloaded = download_all_papers(
                [p for p in papers if p.get("pdf_url")],
                dl_folder,
            )
            print(f"\nDiscovered {len(papers)} papers, downloaded {len(downloaded)} PDFs")
            print(f"PDFs saved to: {args.download_folder}")
            print(f"Paper index saved to: discovered_papers.json")
        return

    api_key = args.api_key or os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        logger.error("No API key provided.")
        sys.exit(1)

    try:
        df = process_papers(
            paper_source=args.source,
            input_folder=args.input_folder or "",
            download_folder=args.download_folder,
            output_file=args.output,
            inspire_query=args.inspire_query,
            arxiv_search_query=args.arxiv_query,
            max_papers_per_api=args.max_papers,
            api_key=api_key,
            model=args.model,
            max_pages=args.max_pages,
            max_tokens_per_chunk=args.max_tokens,
            delay_between_papers=args.delay,
        )
        if not df.empty:
            print(f"\nExtracted {len(df)} datasets from {df['paper'].nunique()} papers")
            print(f"Saved to: {args.output}")
        else:
            print("No datasets extracted.")
            sys.exit(1)
    except Exception as e:
        logger.error(f"Failed: {e}")
        sys.exit(1)


if __name__ == "__main__":
    if len(sys.argv) > 1 and not any(a.startswith('--f=') for a in sys.argv):
        main()
    else:
        try:
            import nest_asyncio
            nest_asyncio.apply()
        except ImportError:
            pass
        run_with_config()